In [0]:
from pyspark.sql.utils import AnalysisException
from datetime import date
from pyspark.sql.functions import col, from_unixtime

table_name = "dbt_job_market.default.linkedin_data"

# Use widgets to get the job parameter
trigger_file_path = dbutils.widgets.get("input_file_path")

files = dbutils.fs.ls(trigger_file_path)
json_files = [f.path for f in files if f.path.endswith(".json")]
if not json_files:
    raise ValueError("No JSON files found in input directory.")

latest_file = max(json_files, key=lambda x: dbutils.fs.ls(x)[0].modificationTime)

df = spark.read.format("json") \
    .option("header", "true") \
    .load(latest_file)


# if not trigger_file_path:
#     raise ValueError("Trigger file path not found in job context.")

# df = spark.read.format("json") \
#     .option("header", "true") \
#     .load(trigger_file_path)

df = df.withColumn("scraped_at", from_unixtime(col("scraped_at")).cast("timestamp"))

df.write.mode("append").saveAsTable(table_name)